# Test: test-time fine-tuning on VOT-LT2020

Runs STARK-ST with test-time fine-tuning of the classification head on VOT-LT2020 (vot-toolkit) and evaluates it
with COCO-style metrics. Details: [docs/test.md](../docs/test.md)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # Colab only: get / update the code
    os.system("git clone -q https://github.com/balk21/stark-cls-finetune /content/stark-cls-finetune || git -C /content/stark-cls-finetune pull -q")
    os.chdir("/content/stark-cls-finetune/notebooks")
sys.path.insert(0, os.getcwd())
import importlib, nbhelper; nb = importlib.reload(nbhelper)
nb.init(colab_drive="/content/drive/MyDrive/LOKAP")  # finds / creates the vot1 environment; colab_drive: Drive folder (Colab only)

## Data

In [ ]:
nb.download_checkpoints(["baseline_R101"])  # official STARK-ST101 | also "baseline" (ST50), "baseline_R101_got10k_only", "baseline_got10k_only"
nb.cli("smoke", "--frames", "50")           # quick check without vot-toolkit (mean IoU > 0.5 = OK); downloads ballet (57 MB) if needed
# nb.download_vot_dataset("all")            # optional: all 50 sequences in advance (17.6 GB); a test downloads the ones it uses

## Parameters

In [ ]:
PARAMS = dict(
    name=None,                     # output folder <outputs>/<name>; None = generated from the parameters
    model="stark_st",              # "stark_st" | "stark_s" (no confidence score: ft_mode must be "none")
    model_config="baseline_R101",  # "baseline_R101" (ST101) | "baseline" (ST50) | "baseline_R101_got10k_only" | "baseline_got10k_only"
    checkpoint=None,               # None = official | file name in checkpoints/stark_st2/<model_config>/ | path | "train:<run name>"
    sequences="all",               # "all" (50 sequences, 17.6 GB) | ["ballet", "bull", ...]; missing ones are downloaded
    update_interval=100,           # template update attempt every N frames; 99999 = never
    update_conf_thr=0.5,           # update only if the score is above this
    max_template_updates=-1,       # per sequence; -1 = unlimited
    ft_mode="online",              # "none": plain STARK-ST | "init": first frame | "online": first frame + every template update
    ft_samples="pos",              # "pos": positive sample only | "posneg": + a negative region of the same frame
    ft_lr=1e-4,                    # fine-tuning learning rate (AdamW)
    ft_epochs_init=15,             # steps on the first frame
    ft_epochs_online=1,            # steps per online fine-tuning
    ft_weight_decay=1e-4,
    ft_grad_clip_norm=0.1,
    ft_pos_jitter=True,            # True: jitter the positive box as in STARK stage-2 training | False: exact box
    ft_center_jitter=4.5,          # jitter strength (STARK values)
    ft_scale_jitter=0.5,
    max_ft_updates=-1,             # online fine-tunings per sequence; -1 = unlimited
    seed=0,                        # same seed = same result (on the same GPU type)
    eval_score_thr=0.35,           # fixed score threshold for P / R / F1
    eval_iou_thr=0.5,              # IoU of a correct detection (P / R / F1)
    eval_thr_resolution=100,       # candidate thresholds in the F-max search (vot-toolkit: 100)
    run_redetection=False,         # True: also the VOT-LT "redetection" experiment (~2x time)
    tracker_timeout=300,           # seconds per tracker response
)
nb.describe(PARAMS)  # checks the parameters; shows output folder, checkpoint, sequences to download

## Run

In [ ]:
OUT = nb.test(PARAMS)  # same name + parameters again: resumes (skips finished sequences) | overwrite=True: starts again

## Results

In [ ]:
nb.show_summary(OUT)                      # mAP / AP50 / AP75, F-max threshold, fixed threshold, per sequence
nb.show_f_curve(OUT)                      # P / R / F vs. threshold and the PR curve
nb.show_plots(OUT, sequences=["ballet"])  # IoU and score vs. frame, fine-tuning loss | sequences=None: first 5

## Optional

In [ ]:
# nb.analyze(OUT, score_thr=0.5, iou_thr=0.5)                             # metrics again with other evaluation settings (no tracking)
# nb.test_many([dict(PARAMS, ft_mode=m) for m in ("none", "init", "online")])  # several experiments one after another